In [ ]:
import sys
from pathlib import Path
INPUT_ROOT = Path('/kaggle/input')
OUT_DIR = Path('/kaggle/working')
_SKIP = {'train_series', 'test_series'}

def find_dir_containing(marker, root=INPUT_ROOT, max_depth=8):
    """BFS for a directory containing `marker`, pruning the huge series trees."""
    frontier = [root]
    for _ in range(max_depth + 1):
        nxt = []
        for c in frontier:
            if (c / marker).exists():
                return c
            try:
                nxt += [d for d in c.iterdir() if d.is_dir() and d.name not in _SKIP]
            except OSError:
                pass
        frontier = nxt
    raise FileNotFoundError('not found: ' + marker)

SRC = find_dir_containing('rsna_knee/__init__.py'); sys.path.insert(0, str(SRC))
COMP = find_dir_containing('sample_submission.csv')
DINOV2 = find_dir_containing('preprocessor_config.json')
print('src:', SRC); print('comp:', COMP); print('dinov2:', DINOV2)

In [ ]:
import time
import numpy as np, pandas as pd, torch
from rsna_knee.data.series import assign_slots
from rsna_knee.data.label_expansion import add_derived_confidence, build_training_targets
from rsna_knee.data.labels import load_train_labels
from rsna_knee.datasets import StudyDataset, compute_safe_max_cache_studies
from rsna_knee.metrics import LABEL_COLUMNS, per_label_auc
from rsna_knee.models.vision import KneeMultiLabelModel
from rsna_knee.train import train_one_epoch
from rsna_knee.infer import run_inference
from rsna_knee.validate import community_holdout_ids

def select_device():
    if not torch.cuda.is_available():
        return 'cpu'
    try:
        (torch.zeros(1, device='cuda') + 1).cpu(); return 'cuda'
    except Exception as e:
        print('cuda failed sanity op:', e); return 'cpu'

device = select_device(); print('device:', device)

In [ ]:
# Recipe replicates base v2 EXACTLY (DINOv2-base, per-slot attention head, K=1).
# Deliberate: our architecture differs from the public DINOv3 member (which uses
# token cross-attention over 16 slices), and that difference is WHY blending helped
# (+0.0063 upper bound at w=0.30). Converging our design toward theirs would reduce
# the decorrelation that produces the gain.
BATCH_SIZE = 8
EPOCHS = 6            # base v2's total epoch count; 9 overfit (tier1 0.7931 -> 0.7715)
N_UNFROZEN_BLOCKS = 2
ENCODER_LR = 1e-5
HEAD_LR = 1e-3

train_labels = load_train_labels(COMP / 'train.csv')
train_series_csv = pd.read_csv(COMP / 'train_series.csv')

SILVER = find_dir_containing('llm_labels_v4_blend.csv')
silver = add_derived_confidence(pd.read_csv(SILVER / 'llm_labels_v4_blend.csv'))
targets = build_training_targets(train_labels, silver)

# THEIR split: train on groups 1-4, hold out group 0. This is the whole point --
# it makes our member honestly evaluable on the same 881 studies their members were
# scored on, and legitimately composable with them.
train_ids, holdout_ids = community_holdout_ids(train_labels)
print('train:', len(train_ids), ' holdout:', len(holdout_ids), '(expect 3526 / 881)')
assert len(holdout_ids) == 881, 'community split did not reproduce'

series_dir = COMP / 'train_series'
tr_slots = assign_slots(train_series_csv[train_series_csv.StudyInstanceUID.isin(set(train_ids))], series_dir)

NUM_WORKERS = 4
cache = compute_safe_max_cache_studies(NUM_WORKERS)
print('max_cache_studies/worker:', cache)
ds = StudyDataset(train_ids, tr_slots, series_dir, targets, max_cache_studies=cache)
dl = torch.utils.data.DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                                 num_workers=NUM_WORKERS, persistent_workers=True)
print('train dataset size:', len(ds))

In [ ]:
model = KneeMultiLabelModel(num_slots=6, num_labels=12, hf_pretrained_path=str(DINOV2)).to(device)
model.encoder.freeze_except_last_n_blocks(N_UNFROZEN_BLOCKS)
n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_all = sum(p.numel() for p in model.parameters())
print('trainable %.1f%%' % (100.0 * n_tr / n_all))
opt = torch.optim.AdamW(model.discriminative_param_groups(ENCODER_LR, HEAD_LR))
for ep in range(EPOCHS):
    t0 = time.time()
    loss = train_one_epoch(model, dl, opt, device)
    print('epoch %d/%d loss=%.4f (%.0fs)' % (ep + 1, EPOCHS, loss, time.time() - t0))
torch.save(model.state_dict(), OUT_DIR / 'model_community_split_v1.pt')
print('checkpoint saved')

In [ ]:
# Evaluate on THEIR 881 holdout -- studies this model never trained on, and the
# exact same set the public member scored 0.9178 on. Directly comparable.
ho_slots = assign_slots(train_series_csv[train_series_csv.StudyInstanceUID.isin(set(holdout_ids))], series_dir)
t0 = time.time()
pred = run_inference(model, holdout_ids, ho_slots, series_dir, device, batch_size=BATCH_SIZE)
print('inference %.1f min' % ((time.time() - t0) / 60))
pred = pred.set_index('StudyInstanceUID')
pred.to_csv(OUT_DIR / 'holdout_predictions.csv')

tg = targets[targets.StudyInstanceUID.isin(set(holdout_ids))].set_index('StudyInstanceUID')
y = (tg.loc[pred.index, LABEL_COLUMNS] >= 0.5).astype(float)
sc = per_label_auc(y.reset_index(drop=True), pred.reset_index(drop=True))
vals = [v for v in sc.values() if not np.isnan(v)]
print('')
print('=== OUR MEMBER ON THEIR CLEAN 881 HOLDOUT (vs silver) ===')
for k, v in sorted(sc.items(), key=lambda kv: kv[1]):
    print('   %-20s %.3f' % (k, v))
print('   MACRO = %.4f' % float(np.mean(vals)))
print('')
print('reference: public DINOv3 member = 0.9178 on this same set')
print('           base v2 (contaminated on 706 of these) = 0.8854')
print('           governing target = 0.94')
print('holdout_predictions.csv written for offline blend weight search')